# Data Preprocessing

## Objective

The production dataset has successfully passed the validation stage.

The objective of this notebook is to prepare the dataset for feature engineering and machine learning.

The preprocessing pipeline includes:

- Loading the validated dataset
- Selecting the project language
- Handling missing values
- Combining text fields
- Removing unnecessary columns
- Cleaning textual data
- Saving the processed dataset

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

pd.set_option("display.max_columns", None)

## Load Dataset

Load the validated production support ticket dataset.

In [2]:
DATA_PATH = Path("../data/raw/tickets_dataset.csv")

df = pd.read_csv(DATA_PATH)

print(df.shape)

df.head()

(28587, 16)


,subject,body,answer,type,queue,priority,language,version,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8
0,Wesentlicher Sicherheitsvorfall,"Sehr geehrtes Support-Team,\n\nich möchte eine...",Vielen Dank für die Meldung des kritischen Sic...,Incident,Technical Support,high,de,51,Security,Outage,Disruption,Data Breach,NaN,NaN,NaN,NaN
1,Account Disruption,"Dear Customer Support Team,\n\nI am writing to...","Thank you for reaching out, <name>. We are awa...",Incident,Technical Support,high,en,51,Account,Disruption,Outage,IT,Tech Support,NaN,NaN,NaN
2,Query About Smart Home System Integration Feat...,"Dear Customer Support Team,\n\nI hope this mes...",Thank you for your inquiry. Our products suppo...,Request,Returns and Exchanges,medium,en,51,Product,Feature,Tech Support,NaN,NaN,NaN,NaN,NaN
3,Inquiry Regarding Invoice Details,"Dear Customer Support Team,\n\nI hope this mes...",We appreciate you reaching out with your billi...,Request,Billing and Payments,low,en,51,Billing,Payment,Account,Documentation,Feedback,NaN,NaN,NaN
4,Question About Marketing Agency Software Compa...,"Dear Support Team,\n\nI hope this message reac...",Thank you for your inquiry. Our product suppor...,Problem,Sales and Pre-Sales,medium,en,51,Product,Feature,Feedback,Tech Support,NaN,NaN,NaN,NaN


## Select Project Language

The dataset contains both English and German tickets.

Version 1 of QResolve focuses on English support tickets to simplify preprocessing and establish a strong baseline model.

Multilingual support can be added in a future version.

In [3]:
df = df[df["language"] == "en"].copy()

print(df.shape)

df.head()

(16338, 16)


,subject,body,answer,type,queue,priority,language,version,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8
1,Account Disruption,"Dear Customer Support Team,\n\nI am writing to...","Thank you for reaching out, <name>. We are awa...",Incident,Technical Support,high,en,51,Account,Disruption,Outage,IT,Tech Support,NaN,NaN,NaN
2,Query About Smart Home System Integration Feat...,"Dear Customer Support Team,\n\nI hope this mes...",Thank you for your inquiry. Our products suppo...,Request,Returns and Exchanges,medium,en,51,Product,Feature,Tech Support,NaN,NaN,NaN,NaN,NaN
3,Inquiry Regarding Invoice Details,"Dear Customer Support Team,\n\nI hope this mes...",We appreciate you reaching out with your billi...,Request,Billing and Payments,low,en,51,Billing,Payment,Account,Documentation,Feedback,NaN,NaN,NaN
4,Question About Marketing Agency Software Compa...,"Dear Support Team,\n\nI hope this message reac...",Thank you for your inquiry. Our product suppor...,Problem,Sales and Pre-Sales,medium,en,51,Product,Feature,Feedback,Tech Support,NaN,NaN,NaN,NaN
5,Feature Query,"Dear Customer Support,\n\nI hope this message ...",Thank you for your inquiry. Please specify whi...,Request,Technical Support,high,en,51,Feature,Product,Documentation,Feedback,NaN,NaN,NaN,NaN


## Remove Unnecessary Columns

Several columns are not required for ticket priority prediction.

These include:

- answer
- language
- version

These columns either contain response information that would not be available at prediction time or metadata that is not useful for the current model.

In [4]:
columns_to_drop = [
    "answer",
    "language",
    "version"
]

df.drop(columns=columns_to_drop, inplace=True)

df.head()

,subject,body,type,queue,priority,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8
1,Account Disruption,"Dear Customer Support Team,\n\nI am writing to...",Incident,Technical Support,high,Account,Disruption,Outage,IT,Tech Support,NaN,NaN,NaN
2,Query About Smart Home System Integration Feat...,"Dear Customer Support Team,\n\nI hope this mes...",Request,Returns and Exchanges,medium,Product,Feature,Tech Support,NaN,NaN,NaN,NaN,NaN
3,Inquiry Regarding Invoice Details,"Dear Customer Support Team,\n\nI hope this mes...",Request,Billing and Payments,low,Billing,Payment,Account,Documentation,Feedback,NaN,NaN,NaN
4,Question About Marketing Agency Software Compa...,"Dear Support Team,\n\nI hope this message reac...",Problem,Sales and Pre-Sales,medium,Product,Feature,Feedback,Tech Support,NaN,NaN,NaN,NaN
5,Feature Query,"Dear Customer Support,\n\nI hope this message ...",Request,Technical Support,high,Feature,Product,Documentation,Feedback,NaN,NaN,NaN,NaN


In [5]:
df.columns.tolist()

['subject',
 'body',
 'type',
 'queue',
 'priority',
 'tag_1',
 'tag_2',
 'tag_3',
 'tag_4',
 'tag_5',
 'tag_6',
 'tag_7',
 'tag_8']

## Handle Missing Subject Values

Some tickets do not contain a subject.

Instead of removing these tickets, missing subjects are replaced with an empty string so that every ticket can still be used during model training.

In [6]:
df["subject"] = df["subject"].fillna("")

print("Missing subjects:", df["subject"].isna().sum())

Missing subjects: 0


## Create Combined Ticket Text

The ticket subject and ticket body are merged into a single text feature.

This allows the machine learning model to learn from both the summary and the detailed description of each support ticket.

In [7]:
df["text"] = (
    df["subject"].str.strip()
    + " "
    + df["body"].str.strip()
)

df["text"] = df["text"].str.strip()

df[["subject", "body", "text"]].head()

,subject,body,text
1,Account Disruption,"Dear Customer Support Team,\n\nI am writing to...","Account Disruption Dear Customer Support Team,..."
2,Query About Smart Home System Integration Feat...,"Dear Customer Support Team,\n\nI hope this mes...",Query About Smart Home System Integration Feat...
3,Inquiry Regarding Invoice Details,"Dear Customer Support Team,\n\nI hope this mes...",Inquiry Regarding Invoice Details Dear Custome...
4,Question About Marketing Agency Software Compa...,"Dear Support Team,\n\nI hope this message reac...",Question About Marketing Agency Software Compa...
5,Feature Query,"Dear Customer Support,\n\nI hope this message ...","Feature Query Dear Customer Support,\n\nI hope..."


## Remove Original Text Columns

The original subject and body columns are removed after creating the combined text feature.

The new `text` column becomes the primary input for the NLP pipeline.

In [8]:
df.drop(columns=["subject", "body"], inplace=True)

print(df.columns.tolist())
print(df.shape)

['type', 'queue', 'priority', 'tag_1', 'tag_2', 'tag_3', 'tag_4', 'tag_5', 'tag_6', 'tag_7', 'tag_8', 'text']
(16338, 12)


## Basic Text Cleaning

Clean the combined ticket text by:

- converting to lowercase
- removing extra whitespace
- removing line breaks

The goal is to standardize the text while preserving its semantic meaning.

In [9]:
import re

In [10]:
def clean_text(text):
    text = str(text)

    # lowercase
    text = text.lower()

    # remove line breaks
    text = text.replace("\n", " ")

    # collapse multiple spaces
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [11]:
df["text"] = df["text"].apply(clean_text)

df[["text"]].head()

,text
1,"account disruption dear customer support team,..."
2,query about smart home system integration feat...
3,inquiry regarding invoice details dear custome...
4,question about marketing agency software compa...
5,"feature query dear customer support,\n\ni hope..."


## Final Dataset Validation

Verify that no missing values remain in the text or target columns before saving the processed dataset.

In [12]:
print("Missing text:", df["text"].isna().sum())
print("Missing priority:", df["priority"].isna().sum())

Missing text: 0
Missing priority: 0


## Save Processed Dataset

Store the cleaned dataset for feature engineering and model training.

In [13]:
PROCESSED_PATH = Path("../data/processed")

PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

In [14]:
OUTPUT_FILE = PROCESSED_PATH / "tickets_processed.csv"

df.to_csv(OUTPUT_FILE, index=False)

print("Saved to:", OUTPUT_FILE)

Saved to: ..\data\processed\tickets_processed.csv


In [15]:
processed_df = pd.read_csv(OUTPUT_FILE)

processed_df.head()

,type,queue,priority,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8,text
0,Incident,Technical Support,high,Account,Disruption,Outage,IT,Tech Support,NaN,NaN,NaN,"account disruption dear customer support team,..."
1,Request,Returns and Exchanges,medium,Product,Feature,Tech Support,NaN,NaN,NaN,NaN,NaN,query about smart home system integration feat...
2,Request,Billing and Payments,low,Billing,Payment,Account,Documentation,Feedback,NaN,NaN,NaN,inquiry regarding invoice details dear custome...
3,Problem,Sales and Pre-Sales,medium,Product,Feature,Feedback,Tech Support,NaN,NaN,NaN,NaN,question about marketing agency software compa...
4,Request,Technical Support,high,Feature,Product,Documentation,Feedback,NaN,NaN,NaN,NaN,"feature query dear customer support,\n\ni hope..."


# Preprocessing Summary

The production support ticket dataset has been successfully preprocessed.

Completed steps:

- Selected English-language tickets
- Handled missing subject values
- Combined subject and body into a unified text feature
- Removed unused columns
- Standardized text formatting
- Saved the processed dataset for feature engineering

The dataset is now ready for vectorization and model development.